## 1. Imports et configuration

In [ ]:
import os, re, json, math, random, hashlib, unicodedata, platform, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)

STEP2_DIR = Path("outputs/step2_index")          # produit par le notebook de l'étape 2
SEED_PATH = Path("train_seed_b1.jsonl")
TEST_SET_PATH = Path("test_set_b1.jsonl")
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step3_dataset"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_FRACTION = 0.20            # part des questions-sources réservée à la validation (split PAR QUESTION, jamais par exemple)
N_VARIANTS_TRAIN = 2           # nombre de contextes différents générés par question d'entraînement
MISS_RATE = 0.15               # part des questions « avec réponse » dont on génère AUSSI une variante « retrieval manqué » (réponse absente du contexte -> abstention)
MISS_ANSWER_OVERLAP = 0.5       # une variante « miss » est écartée si ≥ 50 % du contenu de la réponse est encore dans un passage du contexte (le guide répète certains faits)
ABSTAIN_MARKER = "INFORMATION ABSENTE"          # identique à l'étape 1 (les métriques d'abstention en dépendent)
ABSTAIN_ANSWER = ABSTAIN_MARKER + " : les passages fournis ne contiennent pas cette information."
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

random.seed(SEED); np.random.seed(SEED)
rng = random.Random(SEED)
print("python", platform.python_version())

## 2. Utilitaires (identiques à l'étape 2 : normalisation, BM25, citation)

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", s.lower().replace("’", "'"))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s  ]+", " ", s).strip()

STOPWORDS = set("le la les l un une des de du d et ou a au aux en dans par pour sur sous avec sans que qui quoi est sont etre ce cet cette ces se sa son ses "
                "leur leurs ne pas plus il ils elle elles on y c s n qu j m t si comme mais donc car faut faire peut doit quel quelle quels quelles".split())
def tokenize(text):
    toks = re.findall(r"[a-z0-9]+", norm(text))
    return [t[:-1] if len(t) > 3 and t.endswith("s") else t for t in toks if t not in STOPWORDS]

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [Counter(d) for d in docs_tokens]
        self.len = np.array([len(d) for d in docs_tokens], dtype=float); self.avg = self.len.mean()
        df = Counter(t for d in self.docs for t in d); n = len(docs_tokens)
        self.idf = {t: math.log(1 + (n - f + 0.5) / (f + 0.5)) for t, f in df.items()}
    def scores(self, q_tokens):
        s = np.zeros(len(self.docs))
        for t in set(q_tokens):
            if t in self.idf:
                tf = np.array([d.get(t, 0) for d in self.docs], dtype=float)
                s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.len / self.avg))
        return s

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

def cite(c):
    p = [x for x in c["pages"] if x >= 1]            # la page de présentation du PDF n'a pas de numéro imprimé
    pages = "page non numérotée" if not p else (f"p. {p[0]}" if len(p) == 1 else f"p. {p[0]}-{p[-1]}")
    where = "Présentation du guide" if c["fiche"] == 0 else f"Fiche {c['fiche']} — {c['fiche_title']}"
    return f"{where} — {c['section_label']}, {pages}"

from functools import lru_cache
@lru_cache(maxsize=None)
def rx(pattern):
    """Regex compilée sur le texte normalisé (mise en cache : le scan de fuite l'appelle des centaines de milliers de fois)."""
    return re.compile(norm(pattern))

def jaccard(a, b):
    a, b = set(tokenize(a)), set(tokenize(b))
    return len(a & b) / max(len(a | b), 1)